# 05. Poisson Scoreline Matrix Simulation

**Stage:** 05_simulation  
**Inputs:** `data/processed/adjusted_lambdas.parquet`, `src/config/config.yaml`  
**Outputs:** `data/processed/scoreline_probabilities.parquet`  

This notebook constructs bivariate Poisson scoreline matrices for every fixture from home and away expected goals ($\lambda$), computes 1X2 probabilities (Home/Draw/Away), Over/Under totals (Over 2.5), Both Teams To Score (BTTS), and most likely exact scorelines. Visualizations include sample scoreline heatmaps for high-profile fixtures.

## 1. Environment & Setup

In [ ]:
# Load configuration and imports
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Locate repo root dynamically
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "config" / "loader.py").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.config.loader import load_config, get_max_goals
from src.simulation.scoreline_matrix import (
    build_scoreline_matrix,
    extract_1x2_probabilities,
    extract_over_under_probabilities,
    extract_btts_probability,
    extract_most_likely_scoreline,
    compute_fixture_scoreline_outcomes,
)

config = load_config()
max_goals = get_max_goals(config)
processed_dir = repo_root / "data" / "processed"
input_file = processed_dir / "adjusted_lambdas.parquet"
output_file = processed_dir / "scoreline_probabilities.parquet"

print(f"Loaded config. Max goals bound: {max_goals}")

## 2. Load Adjusted Expected Goal Lambdas

In [ ]:
assert input_file.exists(), f"Required input artifact missing: {input_file}"
df_lambdas = pd.read_parquet(input_file)
assert len(df_lambdas) > 0, "Input adjusted_lambdas.parquet is empty!"

print(f"Loaded {len(df_lambdas)} fixtures from {input_file.name}")
df_lambdas.head(3)

## 3. Sample Scoreline Matrix & Heatmap Visualization

In [ ]:
# NBVAL_IGNORE_OUTPUT
# Pick a representative fixture with valid lambdas
sample_row = df_lambdas.dropna(subset=["adjusted_home_lambda", "adjusted_away_lambda"]).iloc[0]
h_team = sample_row["home_team"]
a_team = sample_row["away_team"]
h_lam = sample_row["adjusted_home_lambda"]
a_lam = sample_row["adjusted_away_lambda"]

matrix = build_scoreline_matrix(h_lam, a_lam, max_goals=max_goals)
probs_1x2 = extract_1x2_probabilities(matrix)
probs_ou = extract_over_under_probabilities(matrix, threshold=2.5)
prob_btts = extract_btts_probability(matrix)
ml_h, ml_a, ml_p = extract_most_likely_scoreline(matrix)

print(f"Sample Fixture: {h_team} ({h_lam:.2f}) vs {a_team} ({a_lam:.2f})")
print(f"1X2 Probabilities -> Home: {probs_1x2['prob_home_win']:.1%}, Draw: {probs_1x2['prob_draw']:.1%}, Away: {probs_1x2['prob_away_win']:.1%}")
print(f"Over 2.5: {probs_ou['prob_over']:.1%}, BTTS: {prob_btts:.1%}")
print(f"Most Likely Scoreline: {ml_h}-{ml_a} ({ml_p:.1%})")

plt.figure(figsize=(7, 6))
sns.heatmap(
    matrix * 100,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    xticklabels=range(max_goals + 1),
    yticklabels=range(max_goals + 1)
)
plt.title(f"Scoreline Probabilities (%) — {h_team} vs {a_team}")
plt.xlabel(f"{a_team} Goals (Away)")
plt.ylabel(f"{h_team} Goals (Home)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Compute Outcome Probabilities Across All Fixtures

In [ ]:
df_outcomes = compute_fixture_scoreline_outcomes(df_lambdas, max_goals=max_goals)

valid_outcomes = df_outcomes.dropna(subset=["prob_home_win"])
print(f"Successfully computed scoreline probabilities for {len(valid_outcomes)} / {len(df_outcomes)} fixtures.")
print("\nSummary Outcome Distribution:")
print(valid_outcomes[["prob_home_win", "prob_draw", "prob_away_win", "prob_over_2_5", "prob_btts"]].describe())
valid_outcomes.head(5)

## 5. Save Output Artifact

In [ ]:
# NBVAL_IGNORE_OUTPUT
output_file.parent.mkdir(parents=True, exist_ok=True)
df_outcomes.to_parquet(output_file, index=False)
print(f"Saved {len(df_outcomes)} rows with scoreline outcome probabilities to: {output_file}")